In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-chroma \
    langchain-huggingface \
    chromadb \
    sentence-transformers

In [ ]:
import chromadb
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

print("ChromaDB:", chromadb.__version__)
print("Chroma import: OK")
print("HuggingFace embeddings import: OK")

ChromaDB: 1.5.9
Chroma import: OK
HuggingFace embeddings import: OK


In [ ]:
import json
from langchain_core.documents import Document

with open("movies_data_5000.json", "r", encoding="utf-8") as f:
    data = json.load(f)

final_docs = [Document(**item) for item in data]
final_docs[:4]

[Document(metadata={'title': ' Judgment Night', 'source': 'Documents/movie_tmdb.csv', 'imdb_rating': ' 6.6', 'poster_path': ' /3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg', 'genre': ' Action, Crime, Thriller', 'keywords': ' drug dealer|chicago, illinois|escape|one night|boxing'}, page_content="\nTitle:  Judgment Night\n\nGenres:  Action, Crime, Thriller \n\nproduction_companies :   Largo Entertainment, JVC, Universal Pictures\n\nKeywords:  drug dealer|chicago, illinois|escape|one night|boxing\n\nImdb_Rating :  6.6   \n\nTagline:  Don't move. Don't whisper. Don't even breathe.\n\nOverview: \n Four young friends, while taking a shortcut en route to a local boxing match, witness a brutal murder which leaves them running for their lives."),
 Document(metadata={'title': ' Star Wars', 'source': 'Documents/movie_tmdb.csv', 'imdb_rating': ' 8.6', 'poster_path': ' /6FfCtAuVAW8XJjZ7eWeLibRLWTw.jpg', 'genre': ' Adventure, Action, Science Fiction', 'keywords': ' empire|galaxy|rebellion|android|hermit|smugglin

In [ ]:
# We will load the embedding model from huggingface
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "Sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_community.vectorstores import Chroma

# Create Chroma vector store
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="chroma_db",
    collection_name="movies"
)

/tmp/ipykernel_2294/8780500.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
/tmp/ipykernel_2294/8780500.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [ ]:
# Well this part will take time
from tqdm import tqdm # To add progress bar

for i in tqdm(range(0, len(final_docs)),desc="Indexing Movies"):
    vector_store.add_documents([final_docs[i]])

Indexing Movies: 100%|██████████| 5000/5000 [13:18<00:00,  6.26it/s]


In [ ]:
query = """
An 11-year-old boy discovers he is a wizard and begins studying at a magical boarding school.
As he learns magic and forms lasting friendships, he uncovers the truth about his family's past
and confronts a powerful dark wizard.
"""
similar_docs = vector_store.similarity_search(
    query=query,
    k=10
)

In [ ]:
# Extracting the titles of the movies
for doc in similar_docs:
    print(doc.metadata["title"])

 Harry Potter and the Philosopher's Stone
 Harry Potter and the Order of the Phoenix
 The Sword in the Stone
 Harry Potter and the Goblet of Fire
 Bogus
 About a Boy
 Harry Potter and the Prisoner of Azkaban
 The Craft
 Journey to the Beginning of Time
 My Tutor


In [ ]:
TMDB_IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"

def recommend_movies(query, num_movies=10):
    similar_docs = vector_store.similarity_search(
    query=query,
    k=num_movies
    )

    movies = [f"{TMDB_IMAGE_BASE_URL}{doc.metadata['poster_path'].strip()}" for doc in similar_docs]
    return movies

In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# Movie Recommendation System")

    movie_input = gr.Textbox(
        label="Movie Description",
        placeholder="Enter a movie title or description..."
        )

    search_button = gr.Button("Search similar Movies", variant="primary")

    gallery = gr.Gallery(
        label = "Recommended Movies",
        columns=5,
        height=500
    )

    search_button.click(
        fn = recommend_movies,
        inputs = movie_input,
        outputs = gallery
    )

demo.launch()

/tmp/ipykernel_2294/4159249977.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab4db69213420fa3f0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
